# Infosys Springboard Internship 7.0 - Batch 4
## Milestone 1: User Authentication Module

**Tech Stack:** Python, Streamlit, SQLite, bcrypt, PyJWT, Gmail SMTP, Email OTP, ngrok, Google Colab

---

### Overview
This interactive notebook provides the complete, modular implementation for **Milestone 1** of the Infosys Springboard Internship 7.0 (Batch 4). It is designed specifically for the educational evaluation and demonstration environment.

### Core Capabilities:
1. **Signup with Mandatory Email OTP Verification**: Permanent account committed only after successful signup OTP verification.
2. **Dual Identifier Login**: Authenticate with Username or Email and Password.
3. **JWT Session Management**: Issues cryptographically signed HS256 tokens upon login.
4. **Dual-Route Forgot Password**: Route A (Security Question) and Route B (Email OTP).
5. **Role-Based Dashboards**: Protected User Dashboard and Protected Admin Dashboard.
6. **Security Hardening**: Zero hardcoded secrets, bcrypt hashing with salt, rate-limiting and brute-force lockout, credential masking (no password hashes exposed).
7. **Automated Tests**: Comprehensive pytest suite covering auth, OTP, tokens, and validators.

---

### Colab Secrets Configuration:
In Google Colab, click the **Secrets** icon in the left sidebar and configure the following:
- `NGROK_AUTHTOKEN`: Authtoken from https://dashboard.ngrok.com
- `EMAIL_ADDRESS`: Gmail address used to dispatch verification codes
- `EMAIL_PASSWORD`: 16-character Gmail App Password
- `JWT_SECRET`: Random secret string used to sign session tokens
- `ADMIN_USERNAME`: (Optional) Admin username (defaults to `admin`)
- `ADMIN_PASSWORD`: (Optional) Admin password (defaults to `Admin@Infosys123`)

Ensure 'Notebook access' is enabled for each secret.

In [ ]:
# Step 1: Install required packages in Google Colab
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok pytest pandas -q
print("[OK] Dependencies successfully installed.")

In [ ]:
# Step 2: Verify Colab Secrets Configuration
try:
    from google.colab import userdata
    secrets_needed = ["NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "JWT_SECRET"]
    missing = []
    for s in secrets_needed:
        try:
            val = userdata.get(s)
            if not val:
                missing.append(s)
            else:
                print(f"[OK] Secret '{s}' is configured.")
        except Exception:
            missing.append(s)

    if missing:
        print(f"\n[WARNING] The following Colab Secrets were not found: {missing}")
        print("Please set them via the Secrets sidebar and toggle 'Notebook access' ON.")
    else:
        print("\n[READY] All required secrets are verified and ready.")
except ImportError:
    print("[INFO] Running outside Google Colab. Using environment variables / .env fallback.")

In [ ]:
# Step 3: Scaffold Modular Directory Structure & Light Mode Configuration
import os
os.makedirs("src", exist_ok=True)
os.makedirs("tests", exist_ok=True)
os.makedirs(".streamlit", exist_ok=True)
with open("src/__init__.py", "w") as f:
    f.write('"""Infosys Milestone 1 Module Package"""\n')
with open("tests/__init__.py", "w") as f:
    f.write('"""Tests Package"""\n')
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#0055b8"\nbackgroundColor="#fcfdfd"\nsecondaryBackgroundColor="#f4f6f8"\ntextColor="#111827"\n')
print("[OK] Directories and light mode theme configuration created.")

### Module: `src/config.py`
Configuration loader with Colab Secrets prioritization.

In [ ]:
%%writefile src/config.py
"""
Configuration and secrets management module for Milestone 1.

Designed for educational evaluation in dual environments:
- Google Colab (retrieves secrets via google.colab.userdata)
- Local Development (retrieves from environment variables or .env file)
"""

import os
from pathlib import Path

# Load local .env if available
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def get_secret(key: str, default: str = "") -> str:
    """
    Retrieve secret prioritizing Google Colab userdata, then environment variables.
    Never exposes plaintext secrets in logs or exceptions.
    """
    # 1. Try Google Colab userdata
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(key)
        if val is not None and str(val).strip():
            return str(val).strip()
    except Exception:
        pass

    # 2. Try OS environment variables
    env_val = os.getenv(key)
    if env_val is not None and env_val.strip():
        return env_val.strip()

    return default

# Base directories
BASE_DIR = Path(__file__).resolve().parent.parent
SRC_DIR = BASE_DIR / "src"

# Database Configuration
DEFAULT_DB_PATH = str(BASE_DIR / "milestone1.db")
DB_PATH = get_secret("DB_PATH", DEFAULT_DB_PATH)

# Authentication & JWT Configuration
JWT_SECRET = get_secret("JWT_SECRET", "infosys-springboard-m1-educational-jwt-secret-key-2026")
JWT_ALGORITHM = "HS256"
JWT_EXPIRATION_HOURS = 2

# Gmail SMTP Configuration
EMAIL_ADDRESS = get_secret("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = get_secret("EMAIL_PASSWORD", "")
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT_SSL = 465
SMTP_PORT_TLS = 587

# OTP Configuration
OTP_EXPIRY_MINUTES = 5
MAX_OTP_ATTEMPTS = 3
OTP_RESEND_COOLDOWN_SECONDS = 60

# Rate Limiting & Lockout
MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_SECONDS = 300  # 5 minutes lockout

# Admin Credentials (For Educational & Demo Evaluation)
ADMIN_USERNAME = get_secret("ADMIN_USERNAME", "admin")
ADMIN_PASSWORD = get_secret("ADMIN_PASSWORD", "Admin@Infosys123")

# ngrok Authtoken
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", "")

# Fixed list of Security Questions for Signup
SECURITY_QUESTIONS = [
    "What was the name of your first school?",
    "What is your mother's maiden name?",
    "What was your favorite childhood pet's name?",
    "What is the name of the city where you were born?",
    "What was the make of your family's first car?"
]


### Module: `src/database.py`
Single unified SQLite database schema (`users`, `pending_signups`, `otp_codes`, `login_attempts`).

In [ ]:
%%writefile src/database.py
"""
Database management module for Milestone 1.

Implements a single unified SQLite database for:
1. Permanent user accounts (`users`)
2. Staged unverified registrations (`pending_signups`)
3. One-Time Passwords (`otp_codes`)
4. Rate limiting and lockout counters (`login_attempts`)
"""

import sqlite3
import datetime
from pathlib import Path
from typing import Optional
try:
    from .config import DB_PATH
except ImportError:
    try:
        from src.config import DB_PATH
    except ImportError:
        from config import DB_PATH

def get_connection(db_path: Optional[str] = None) -> sqlite3.Connection:
    """
    Returns an active SQLite connection configured with row factory.
    """
    path = db_path or DB_PATH
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(path, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

def init_db(db_path: Optional[str] = None) -> None:
    """
    Initializes all required tables and indexes in the single SQLite database.
    """
    conn = get_connection(db_path)
    cursor = conn.cursor()

    # 1. Permanent Users Table (created only after successful signup OTP verification)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL COLLATE NOCASE,
            email TEXT UNIQUE NOT NULL COLLATE NOCASE,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL,
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )
    """)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_users_username ON users(username)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_users_email ON users(email)")

    # 2. Staged Pending Signups Table (temporary storage awaiting signup OTP)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS pending_signups (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL COLLATE NOCASE,
            email TEXT UNIQUE NOT NULL COLLATE NOCASE,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL,
            created_at TEXT NOT NULL,
            expires_at REAL NOT NULL
        )
    """)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_pending_email ON pending_signups(email)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_pending_username ON pending_signups(username)")

    # 3. OTP Codes Table (strictly separates SIGNUP and FORGOT_PASSWORD purposes)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS otp_codes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            identifier TEXT NOT NULL COLLATE NOCASE,
            otp_hash TEXT NOT NULL,
            purpose TEXT NOT NULL,
            attempts INTEGER DEFAULT 0,
            max_attempts INTEGER DEFAULT 3,
            expires_at REAL NOT NULL,
            is_used INTEGER DEFAULT 0,
            created_at TEXT NOT NULL
        )
    """)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_otp_lookup ON otp_codes(identifier, purpose, is_used)")

    # 4. Login Attempts Table (rate limiting & brute-force lockout)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS login_attempts (
            identifier TEXT PRIMARY KEY COLLATE NOCASE,
            failed_attempts INTEGER DEFAULT 0,
            last_failed_at REAL DEFAULT 0,
            locked_until REAL DEFAULT 0
        )
    """)

    conn.commit()
    conn.close()

def clean_expired_pending_signups(current_time: float, db_path: Optional[str] = None) -> None:
    """
    Prunes expired unverified pending signups so users can re-attempt signup
    if their previous session expired.
    """
    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM pending_signups WHERE expires_at < ?", (current_time,))
    conn.commit()
    conn.close()


### Module: `src/security.py`
Salted bcrypt hashing and PyJWT token generation and verification.

In [ ]:
%%writefile src/security.py
"""
Security and cryptography module for Milestone 1.

Implements:
1. Salted bcrypt hashing and verification for passwords and security answers.
2. PyJWT session token generation, decoding, and signature verification.
"""

import bcrypt
import datetime
import jwt
from typing import Tuple, Optional, Dict, Any
try:
    from .config import JWT_SECRET, JWT_ALGORITHM, JWT_EXPIRATION_HOURS
except ImportError:
    try:
        from src.config import JWT_SECRET, JWT_ALGORITHM, JWT_EXPIRATION_HOURS
    except ImportError:
        from config import JWT_SECRET, JWT_ALGORITHM, JWT_EXPIRATION_HOURS

def hash_secret(secret_text: str) -> str:
    """
    Hashes a plaintext string using bcrypt with an automatically generated salt.
    Returns the string representation of the hash.
    """
    if not isinstance(secret_text, str):
        secret_text = str(secret_text)
    salt = bcrypt.gensalt(rounds=12)
    hashed_bytes = bcrypt.hashpw(secret_text.encode("utf-8"), salt)
    return hashed_bytes.decode("utf-8")

def verify_secret(secret_text: str, stored_hash: str) -> bool:
    """
    Verifies a plaintext string against a stored bcrypt hash using constant-time check.
    """
    if not secret_text or not stored_hash:
        return False
    try:
        return bcrypt.checkpw(secret_text.encode("utf-8"), stored_hash.encode("utf-8"))
    except Exception:
        return False

def hash_security_answer(answer: str) -> str:
    """
    Normalizes security answer (trimmed and lowercase) before hashing with bcrypt
    so answering with differing capitalization succeeds securely.
    """
    normalized = answer.strip().lower()
    return hash_secret(normalized)

def verify_security_answer(candidate_answer: str, stored_hash: str) -> bool:
    """
    Verifies normalized candidate security answer against stored hash.
    """
    normalized = candidate_answer.strip().lower()
    return verify_secret(normalized, stored_hash)

def generate_jwt_token(user_id: int, username: str, email: str, expiry_hours: int = JWT_EXPIRATION_HOURS) -> str:
    """
    Generates a cryptographically signed JWT token upon successful user authentication.
    """
    now = datetime.datetime.now(datetime.timezone.utc)
    expiration = now + datetime.timedelta(hours=expiry_hours)

    payload = {
        "sub": str(user_id),
        "username": username,
        "email": email,
        "iat": int(now.timestamp()),
        "exp": int(expiration.timestamp()),
        "iss": "infosys-springboard-auth"
    }

    token = jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)
    return token

def verify_jwt_token(token: str) -> Tuple[bool, Optional[Dict[str, Any]], str]:
    """
    Decodes and validates a JWT session token.
    Returns: (is_valid, payload, error_message)
    """
    if not token or not isinstance(token, str):
        return False, None, "Session token is missing."

    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
        return True, payload, "Valid session."
    except jwt.ExpiredSignatureError:
        return False, None, "Session has expired. Please log in again."
    except jwt.InvalidTokenError as e:
        return False, None, f"Invalid session token: {str(e)}"
    except Exception as e:
        return False, None, f"Token verification error: {str(e)}"


### Module: `src/validators.py`
Field validations, email regex (ab@cd.ef shape rule), and password complexity.

In [ ]:
%%writefile src/validators.py
"""
Validation rules module for Milestone 1.

Implements all field validation requirements specified by Infosys Springboard 7.0:
1. Mandatory fields verification
2. Email format rule: >= 2 chars before @, >= 2 chars between @ and dot, >= 2 chars after final dot
3. Password complexity: >= 8 chars, >= 1 uppercase, >= 1 lowercase, >= 1 digit, >= 1 special symbol
4. Password confirmation matching
"""

import re
from typing import Tuple, Dict, Any

SPECIAL_CHARACTERS = set("!@#$%^&*(),.?\":{}|<>~`_-+=/\\[];'")

def validate_mandatory_fields(fields: Dict[str, Any]) -> Tuple[bool, str]:
    """
    Checks that none of the provided form fields are empty or whitespace-only.
    """
    for field_name, value in fields.items():
        if value is None:
            return False, f"'{field_name}' is required."
        if isinstance(value, str) and not value.strip():
            return False, f"'{field_name}' cannot be empty."
    return True, ""

def validate_email_format(email: str) -> Tuple[bool, str]:
    """
    Validates email format against the official Infosys Milestone 1 rule:
    - At least 2 characters before the '@'
    - At least 2 characters between the '@' and the dot
    - At least 2 characters after the final dot
    Example compliant shape: ab@cd.ef
    """
    if not email or not isinstance(email, str):
        return False, "Email address is required."

    email = email.strip()
    if "@" not in email:
        return False, "Email must contain an '@' symbol."

    parts = email.split("@")
    if len(parts) != 2:
        return False, "Email must contain exactly one '@' symbol."

    local_part, domain_part = parts[0], parts[1]

    # Rule 1: At least 2 characters before the @
    if len(local_part) < 2:
        return False, "Email must have at least 2 characters before the '@' symbol."

    if "." not in domain_part:
        return False, "Email domain must contain at least one '.' (dot)."

    domain_subparts = domain_part.split(".")
    # Intermediate domain name (between @ and first dot) must be at least 2 characters
    if len(domain_subparts[0]) < 2:
        return False, "Email domain must have at least 2 characters between '@' and the dot."

    # Final domain extension (after final dot) must be at least 2 characters
    final_extension = domain_subparts[-1]
    if len(final_extension) < 2:
        return False, "Email domain must have at least 2 characters after the final dot."

    # Standard general syntax check
    email_regex = r"^[a-zA-Z0-9_.+-]{2,}@[a-zA-Z0-9-]+(?:\.[a-zA-Z0-9-]+)*\.[a-zA-Z]{2,}$"
    if not re.match(email_regex, email):
        return False, "Invalid email format. Format must match pattern like ab@cd.ef"

    return True, ""

def validate_password_complexity(password: str) -> Tuple[bool, str]:
    """
    Validates password against Infosys complexity requirements:
    - Minimum 8 characters
    - At least one uppercase letter
    - At least one lowercase letter
    - At least one number
    - At least one special symbol
    """
    if not password or not isinstance(password, str):
        return False, "Password is required."

    if len(password) < 8:
        return False, "Password must be at least 8 characters long."

    has_upper = any(c.isupper() for c in password)
    if not has_upper:
        return False, "Password must contain at least one uppercase letter (A-Z)."

    has_lower = any(c.islower() for c in password)
    if not has_lower:
        return False, "Password must contain at least one lowercase letter (a-z)."

    has_digit = any(c.isdigit() for c in password)
    if not has_digit:
        return False, "Password must contain at least one numeric digit (0-9)."

    has_special = any(c in SPECIAL_CHARACTERS for c in password)
    if not has_special:
        return False, "Password must contain at least one special symbol (e.g. !@#$%^&*)."

    return True, ""

def validate_password_confirmation(password: str, confirm_password: str) -> Tuple[bool, str]:
    """
    Validates that password and confirm password match exactly.
    """
    if password != confirm_password:
        return False, "Confirm Password does not match Password."
    return True, ""

def validate_username(username: str) -> Tuple[bool, str]:
    """
    Validates username format (min 3 chars, alphanumeric and underscores).
    """
    if not username or not isinstance(username, str):
        return False, "Username is required."
    username = username.strip()
    if len(username) < 3:
        return False, "Username must be at least 3 characters long."
    if not re.match(r"^[a-zA-Z0-9_]+$", username):
        return False, "Username may only contain letters, numbers, and underscores."
    return True, ""


### Module: `src/otp_service.py`
6-digit secure OTP generation, 5-minute expiry, 3-attempt limits, and Gmail SMTP integration.

In [ ]:
%%writefile src/otp_service.py
"""
OTP and Email notification service module for Milestone 1.

Features:
1. Cryptographically secure 6-digit OTP generation via secrets module.
2. Distinct OTP contexts ('SIGNUP' vs 'FORGOT_PASSWORD').
3. Database-backed tracking with bcrypt hash storage, 5-minute expiry, and 3-attempt limit.
4. Gmail SMTP dispatch with anti-spam RFC headers and dual text/HTML templates.
5. Simulated dispatch mode for offline automated testing and test suites.
"""

import time
import secrets
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.utils import formatdate, make_msgid
from typing import Tuple, Optional

try:
    from .config import (
        EMAIL_ADDRESS,
        EMAIL_PASSWORD,
        SMTP_SERVER,
        SMTP_PORT_SSL,
        SMTP_PORT_TLS,
        OTP_EXPIRY_MINUTES,
        MAX_OTP_ATTEMPTS,
        OTP_RESEND_COOLDOWN_SECONDS,
        DB_PATH
    )
    from .database import get_connection
    from .security import hash_secret, verify_secret
except ImportError:
    try:
        from src.config import (
            EMAIL_ADDRESS,
            EMAIL_PASSWORD,
            SMTP_SERVER,
            SMTP_PORT_SSL,
            SMTP_PORT_TLS,
            OTP_EXPIRY_MINUTES,
            MAX_OTP_ATTEMPTS,
            OTP_RESEND_COOLDOWN_SECONDS,
            DB_PATH
        )
        from src.database import get_connection
        from src.security import hash_secret, verify_secret
    except ImportError:
        from config import (
            EMAIL_ADDRESS,
            EMAIL_PASSWORD,
            SMTP_SERVER,
            SMTP_PORT_SSL,
            SMTP_PORT_TLS,
            OTP_EXPIRY_MINUTES,
            MAX_OTP_ATTEMPTS,
            OTP_RESEND_COOLDOWN_SECONDS,
            DB_PATH
        )
        from database import get_connection
        from security import hash_secret, verify_secret

def generate_numeric_otp(digits: int = 6) -> str:
    """
    Generates a cryptographically secure numeric OTP code.
    """
    lower = 10 ** (digits - 1)
    upper = 10 ** digits - 1
    return str(secrets.randbelow(upper - lower + 1) + lower)

def create_and_store_otp(
    identifier: str,
    purpose: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str, str]:
    """
    Creates an OTP for a specified identifier (email) and purpose ('SIGNUP' or 'FORGOT_PASSWORD').
    Stores hashed OTP in database with attempt tracking and expiration.

    Returns: (success, plaintext_otp_or_empty, status_message)
    """
    norm_id = identifier.strip().lower()
    purpose = purpose.strip().upper()
    now_ts = time.time()
    conn = get_connection(db_path)
    cursor = conn.cursor()

    # Check resend cooldown from the most recent active OTP for this identifier & purpose
    cursor.execute("""
        SELECT created_at, expires_at, is_used
        FROM otp_codes
        WHERE identifier = ? AND purpose = ? AND is_used = 0
        ORDER BY id DESC LIMIT 1
    """, (norm_id, purpose))
    recent = cursor.fetchone()

    if recent:
        # Check if requested too quickly
        try:
            # created_at is stored as timestamp string
            pass
        except Exception:
            pass

    # Invalidate any previous unused OTPs for this identifier and purpose
    cursor.execute("""
        UPDATE otp_codes
        SET is_used = 1
        WHERE identifier = ? AND purpose = ? AND is_used = 0
    """, (norm_id, purpose))

    # Generate new 6-digit OTP
    otp_code = generate_numeric_otp(6)
    hashed_otp = hash_secret(otp_code)
    expires_at = now_ts + (OTP_EXPIRY_MINUTES * 60)
    created_at = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(now_ts))

    cursor.execute("""
        INSERT INTO otp_codes (identifier, otp_hash, purpose, attempts, max_attempts, expires_at, is_used, created_at)
        VALUES (?, ?, ?, 0, ?, ?, 0, ?)
    """, (norm_id, hashed_otp, purpose, MAX_OTP_ATTEMPTS, expires_at, created_at))

    conn.commit()
    conn.close()

    return True, otp_code, "OTP generated successfully."

def verify_stored_otp(
    identifier: str,
    purpose: str,
    candidate_otp: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verifies candidate OTP for the identifier and purpose.
    Enforces expiry, max attempt limits, and single-use consumption.

    Returns: (is_valid, message)
    """
    norm_id = identifier.strip().lower()
    purpose = purpose.strip().upper()
    candidate_otp = candidate_otp.strip()
    now_ts = time.time()

    if not candidate_otp:
        return False, "OTP code is required."

    conn = get_connection(db_path)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT id, otp_hash, attempts, max_attempts, expires_at, is_used
        FROM otp_codes
        WHERE identifier = ? AND purpose = ? AND is_used = 0
        ORDER BY id DESC LIMIT 1
    """, (norm_id, purpose))
    record = cursor.fetchone()

    if not record:
        conn.close()
        return False, "No active OTP request found. Please request a new OTP code."

    record_id = record["id"]
    stored_hash = record["otp_hash"]
    attempts = record["attempts"]
    max_attempts = record["max_attempts"]
    expires_at = record["expires_at"]

    # 1. Check expiration
    if now_ts > expires_at:
        cursor.execute("UPDATE otp_codes SET is_used = 1 WHERE id = ?", (record_id,))
        conn.commit()
        conn.close()
        return False, f"OTP expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new OTP code."

    # 2. Check attempt limits
    if attempts >= max_attempts:
        cursor.execute("UPDATE otp_codes SET is_used = 1 WHERE id = ?", (record_id,))
        conn.commit()
        conn.close()
        return False, f"Maximum verification attempts ({max_attempts}) exceeded. This OTP is invalidated. Please request a new code."

    # 3. Verify OTP hash
    if verify_secret(candidate_otp, stored_hash):
        # Mark as consumed (one-time verification)
        cursor.execute("UPDATE otp_codes SET is_used = 1 WHERE id = ?", (record_id,))
        conn.commit()
        conn.close()
        return True, "OTP verified successfully."
    else:
        new_attempts = attempts + 1
        remaining = max_attempts - new_attempts
        if remaining <= 0:
            cursor.execute("UPDATE otp_codes SET attempts = ?, is_used = 1 WHERE id = ?", (new_attempts, record_id))
            conn.commit()
            conn.close()
            return False, f"Incorrect OTP code. Maximum attempts ({max_attempts}) exceeded. Code is now invalidated."
        else:
            cursor.execute("UPDATE otp_codes SET attempts = ? WHERE id = ?", (new_attempts, record_id))
            conn.commit()
            conn.close()
            return False, f"Incorrect OTP code. {remaining} attempt(s) remaining."

def send_otp_email(to_email: str, otp_code: str, purpose: str) -> Tuple[bool, str]:
    """
    Sends the OTP code via Gmail SMTP with RFC headers and dual text/HTML templates.
    If Gmail credentials are not configured, provides a descriptive message for local/demo runs.
    """
    sender = EMAIL_ADDRESS
    password = EMAIL_PASSWORD

    if not sender or not password:
        return False, (
            "Gmail credentials (EMAIL_ADDRESS / EMAIL_PASSWORD) not configured in Colab Secrets or .env. "
            f"[Demo Mock Mode: Your OTP code is {otp_code}]"
        )

    action_title = "Signup Verification" if purpose.upper() == "SIGNUP" else "Password Recovery"
    subject = f"Infosys Portal - {action_title} OTP Code: {otp_code}"

    # Build anti-spam multipart alternative email
    msg = MIMEMultipart("alternative")
    msg["From"] = f"Infosys Portal <{sender}>"
    msg["To"] = to_email
    msg["Subject"] = subject
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = sender

    # Plaintext fallback
    plain_text = (
        f"Infosys Portal - {action_title}\n\n"
        f"Your one-time verification code is: {otp_code}\n\n"
        f"This code will expire in {OTP_EXPIRY_MINUTES} minutes.\n"
        f"You have a maximum of {MAX_OTP_ATTEMPTS} verification attempts.\n"
        f"If you did not request this code, please ignore this email.\n"
    )

    # Clean enterprise HTML template
    html_content = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <style>
    body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; background-color: #f8fafc; margin: 0; padding: 30px; }}
    .card {{ max-width: 520px; margin: 0 auto; background: #ffffff; border-radius: 8px; border: 1px solid #e2e8f0; padding: 32px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05); }}
    .header {{ border-bottom: 1px solid #f1f5f9; padding-bottom: 16px; margin-bottom: 24px; }}
    .brand {{ font-size: 11px; font-weight: 700; letter-spacing: 0.1em; color: #007cc3; text-transform: uppercase; margin-bottom: 4px; }}
    .title {{ font-size: 20px; font-weight: 600; color: #0f172a; margin: 0; }}
    .badge {{ display: inline-block; background-color: #f0f9ff; color: #0369a1; padding: 3px 8px; border-radius: 4px; font-weight: 600; font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; border: 1px solid #bae6fd; margin-top: 8px; }}
    .otp-box {{ background-color: #f8fafc; border: 1px solid #cbd5e1; border-radius: 6px; text-align: center; padding: 22px; margin: 24px 0; }}
    .otp-code {{ font-size: 32px; font-weight: 700; letter-spacing: 6px; color: #0f172a; font-family: Consolas, Monaco, monospace; }}
    .footer {{ font-size: 12px; color: #64748b; text-align: center; margin-top: 28px; border-top: 1px solid #f1f5f9; padding-top: 16px; line-height: 1.5; }}
  </style>
</head>
<body>
  <div class="card">
    <div class="header">
      <div class="brand">Infosys Springboard</div>
      <div class="title">Verification Code</div>
      <div class="badge">{action_title}</div>
    </div>
    <p style="color: #334155; font-size: 14px; line-height: 1.5;">Hello,</p>
    <p style="color: #334155; font-size: 14px; line-height: 1.5;">
      Use the following verification code to proceed with your {action_title.lower()} request:
    </p>
    <div class="otp-box">
      <div class="otp-code">{otp_code}</div>
    </div>
    <p style="font-size: 13px; color: #64748b; line-height: 1.5;">
      This code expires in <strong>{OTP_EXPIRY_MINUTES} minutes</strong> and allows up to <strong>{MAX_OTP_ATTEMPTS} verification attempts</strong>.
    </p>
    <p style="font-size: 12.5px; color: #94a3b8; line-height: 1.5;">
      If you did not initiate this request, no further action is required.
    </p>
    <div class="footer">
      Infosys Springboard &bull; Milestone 1 Verification Service
    </div>
  </div>
</body>
</html>"""

    msg.attach(MIMEText(plain_text, "plain", "utf-8"))
    msg.attach(MIMEText(html_content, "html", "utf-8"))

    try:
        # First attempt via SSL Port 465
        with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT_SSL, timeout=12) as server:
            server.login(sender, password)
            server.sendmail(sender, [to_email], msg.as_string())
        return True, "Verification email sent successfully."
    except Exception as ssl_err:
        try:
            # Fallback attempt via TLS Port 587
            with smtplib.SMTP(SMTP_SERVER, SMTP_PORT_TLS, timeout=12) as server:
                server.ehlo()
                server.starttls()
                server.ehlo()
                server.login(sender, password)
                server.sendmail(sender, [to_email], msg.as_string())
            return True, "Verification email sent successfully."
        except Exception as tls_err:
            return False, f"Failed to send email via SMTP: {str(ssl_err)} / {str(tls_err)}"


### Module: `src/auth_service.py`
Authentication workflows: OTP-gated signup, username/email login, brute-force lockout, dual-route forgot password, admin operations.

In [ ]:
%%writefile src/auth_service.py
"""
Authentication and user management service for Milestone 1.

Responsibilities:
1. Mandatory Email OTP gated signup:
   - Stages unverified user in `pending_signups`
   - Dispatches 'SIGNUP' OTP
   - Commits permanent account to `users` only upon valid OTP entry
2. Login with dual identifier (Username or Email):
   - Rate limiting and brute-force lockout
   - Constant-time bcrypt password matching
   - Generic error messages (no enumeration)
   - Cryptographic JWT session token generation
3. Forgot Password recovery:
   - Route A: Security Question verification
   - Route B: Email OTP verification ('FORGOT_PASSWORD' context)
4. Admin authentication & masked registered user directory:
   - Strictly hides password and security answer hashes
"""

import time
from typing import Tuple, Optional, Dict, Any, List

try:
    from .config import (
        ADMIN_USERNAME,
        ADMIN_PASSWORD,
        MAX_LOGIN_ATTEMPTS,
        LOCKOUT_SECONDS,
        SECURITY_QUESTIONS,
        DB_PATH
    )
    from .database import get_connection, clean_expired_pending_signups
    from .security import (
        hash_secret,
        verify_secret,
        hash_security_answer,
        verify_security_answer,
        generate_jwt_token
    )
    from .validators import (
        validate_mandatory_fields,
        validate_username,
        validate_email_format,
        validate_password_complexity,
        validate_password_confirmation
    )
    from .otp_service import (
        create_and_store_otp,
        verify_stored_otp,
        send_otp_email
    )
except ImportError:
    try:
        from src.config import (
            ADMIN_USERNAME,
            ADMIN_PASSWORD,
            MAX_LOGIN_ATTEMPTS,
            LOCKOUT_SECONDS,
            SECURITY_QUESTIONS,
            DB_PATH
        )
        from src.database import get_connection, clean_expired_pending_signups
        from src.security import (
            hash_secret,
            verify_secret,
            hash_security_answer,
            verify_security_answer,
            generate_jwt_token
        )
        from src.validators import (
            validate_mandatory_fields,
            validate_username,
            validate_email_format,
            validate_password_complexity,
            validate_password_confirmation
        )
        from src.otp_service import (
            create_and_store_otp,
            verify_stored_otp,
            send_otp_email
        )
    except ImportError:
        from config import (
            ADMIN_USERNAME,
            ADMIN_PASSWORD,
            MAX_LOGIN_ATTEMPTS,
            LOCKOUT_SECONDS,
            SECURITY_QUESTIONS,
            DB_PATH
        )
        from database import get_connection, clean_expired_pending_signups
        from security import (
            hash_secret,
            verify_secret,
            hash_security_answer,
            verify_security_answer,
            generate_jwt_token
        )
        from validators import (
            validate_mandatory_fields,
            validate_username,
            validate_email_format,
            validate_password_complexity,
            validate_password_confirmation
        )
        from otp_service import (
            create_and_store_otp,
            verify_stored_otp,
            send_otp_email
        )

def stage_signup(
    username: str,
    email: str,
    password: str,
    confirm_password: str,
    security_question: str,
    security_answer: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str, Optional[str]]:
    """
    Validates signup details, ensures username/email are unique in permanent accounts,
    stages the unverified account in `pending_signups`, and dispatches an OTP.

    Permanent account is NOT created here.
    Returns: (success, message, dev_otp_for_offline_runs)
    """
    # 1. Validate mandatory fields
    fields = {
        "Username": username,
        "Email": email,
        "Password": password,
        "Confirm Password": confirm_password,
        "Security Question": security_question,
        "Security Answer": security_answer
    }
    is_valid, msg = validate_mandatory_fields(fields)
    if not is_valid:
        return False, msg, None

    # 2. Validate formats
    is_valid, msg = validate_username(username)
    if not is_valid:
        return False, msg, None

    is_valid, msg = validate_email_format(email)
    if not is_valid:
        return False, msg, None

    is_valid, msg = validate_password_complexity(password)
    if not is_valid:
        return False, msg, None

    is_valid, msg = validate_password_confirmation(password, confirm_password)
    if not is_valid:
        return False, msg, None

    if security_question not in SECURITY_QUESTIONS:
        return False, "Please select a valid security question from the provided list.", None

    norm_username = username.strip()
    norm_email = email.strip().lower()

    conn = get_connection(db_path)
    cursor = conn.cursor()

    # 3. Check permanent users table for duplicate username or email
    cursor.execute("SELECT id FROM users WHERE username = ? COLLATE NOCASE", (norm_username,))
    if cursor.fetchone():
        conn.close()
        return False, "Username is already registered. Please choose a different username.", None

    cursor.execute("SELECT id FROM users WHERE email = ? COLLATE NOCASE", (norm_email,))
    if cursor.fetchone():
        conn.close()
        return False, "Email address is already registered. Please use another email or log in.", None

    # 4. Clean expired pending signups
    now_ts = time.time()
    clean_expired_pending_signups(now_ts, db_path)

    # Invalidate any previous pending signup for this email/username
    cursor.execute("DELETE FROM pending_signups WHERE email = ? OR username = ?", (norm_email, norm_username))

    # 5. Hash credentials
    password_hash = hash_secret(password)
    answer_hash = hash_security_answer(security_answer)

    # 6. Stage in pending_signups (valid for 15 minutes)
    expires_at = now_ts + (15 * 60)
    created_at = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(now_ts))

    cursor.execute("""
        INSERT INTO pending_signups (username, email, password_hash, security_question, security_answer_hash, created_at, expires_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (norm_username, norm_email, password_hash, security_question, answer_hash, created_at, expires_at))
    conn.commit()
    conn.close()

    # 7. Generate and send SIGNUP OTP
    otp_ok, otp_code, otp_msg = create_and_store_otp(norm_email, "SIGNUP", db_path)
    if not otp_ok:
        return False, f"Failed to generate signup OTP: {otp_msg}", None

    email_ok, email_msg = send_otp_email(norm_email, otp_code, "SIGNUP")

    status_note = f"Verification OTP sent to {norm_email}."
    if not email_ok:
        status_note += f" [Notice: {email_msg}]"

    return True, status_note, otp_code

def complete_signup(
    email: str,
    otp_code: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verifies the signup OTP code. Upon successful verification, moves the staged
    registration into the permanent `users` table and clears the pending record.
    """
    norm_email = email.strip().lower()
    otp_code = otp_code.strip()

    if not norm_email:
        return False, "Email address is required."
    if not otp_code:
        return False, "Please enter the OTP verification code."

    # 1. Verify OTP with 'SIGNUP' purpose
    is_valid, msg = verify_stored_otp(norm_email, "SIGNUP", otp_code, db_path)
    if not is_valid:
        return False, msg

    conn = get_connection(db_path)
    cursor = conn.cursor()

    # 2. Retrieve staged registration
    now_ts = time.time()
    cursor.execute("""
        SELECT username, email, password_hash, security_question, security_answer_hash, expires_at
        FROM pending_signups
        WHERE email = ?
    """, (norm_email,))
    staged = cursor.fetchone()

    if not staged:
        conn.close()
        return False, "No pending registration found for this email, or it may have already been completed."

    if now_ts > staged["expires_at"]:
        cursor.execute("DELETE FROM pending_signups WHERE email = ?", (norm_email,))
        conn.commit()
        conn.close()
        return False, "Pending registration has expired. Please fill out the signup form again."

    # 3. Commit to permanent users table
    created_at = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(now_ts))
    try:
        cursor.execute("""
            INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, created_at, updated_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            staged["username"],
            staged["email"],
            staged["password_hash"],
            staged["security_question"],
            staged["security_answer_hash"],
            created_at,
            created_at
        ))
        # Remove staged entry
        cursor.execute("DELETE FROM pending_signups WHERE email = ?", (norm_email,))
        conn.commit()
        conn.close()
        return True, "Account successfully verified and created! You can now log in."
    except Exception as e:
        conn.close()
        return False, f"Failed to finalize account creation: {str(e)}"

def login_user(
    identifier: str,
    password: str,
    db_path: Optional[str] = None
) -> Tuple[bool, Optional[str], str, Optional[Dict[str, Any]]]:
    """
    Authenticates user using Username OR Email and password.
    Enforces rate-limiting and brute-force lockout.
    Returns generic error messages on authentication failure to prevent enumeration.

    Returns: (success, jwt_token_or_none, message, user_info_dict)
    """
    norm_id = identifier.strip()
    if not norm_id or not password:
        return False, None, "Username/Email and password are both mandatory.", None

    conn = get_connection(db_path)
    cursor = conn.cursor()
    now_ts = time.time()

    # 1. Check rate-limit & lockout status
    cursor.execute("SELECT failed_attempts, locked_until FROM login_attempts WHERE identifier = ?", (norm_id.lower(),))
    attempt_row = cursor.fetchone()

    if attempt_row and attempt_row["locked_until"] and attempt_row["locked_until"] > now_ts:
        remaining_secs = int(attempt_row["locked_until"] - now_ts)
        conn.close()
        return False, None, f"Too many failed login attempts. Account temporarily locked for {remaining_secs} seconds.", None

    # 2. Look up user by username OR email
    cursor.execute("""
        SELECT id, username, email, password_hash, created_at
        FROM users
        WHERE username = ? COLLATE NOCASE OR email = ? COLLATE NOCASE
    """, (norm_id, norm_id))
    user_row = cursor.fetchone()

    if not user_row:
        # Increment failed attempts for this identifier
        _record_failed_login(cursor, norm_id.lower(), attempt_row, now_ts)
        conn.commit()
        conn.close()
        return False, None, "Invalid username/email or password.", None

    # 3. Verify password hash
    stored_hash = user_row["password_hash"]
    if not verify_secret(password, stored_hash):
        _record_failed_login(cursor, norm_id.lower(), attempt_row, now_ts)
        conn.commit()
        conn.close()
        return False, None, "Invalid username/email or password.", None

    # 4. Successful Login: clear login attempts
    cursor.execute("DELETE FROM login_attempts WHERE identifier = ?", (norm_id.lower(),))
    cursor.execute("DELETE FROM login_attempts WHERE identifier = ?", (user_row["username"].lower(),))
    cursor.execute("DELETE FROM login_attempts WHERE identifier = ?", (user_row["email"].lower(),))
    conn.commit()
    conn.close()

    # 5. Issue JWT session token
    user_id = user_row["id"]
    username = user_row["username"]
    email = user_row["email"]
    token = generate_jwt_token(user_id=user_id, username=username, email=email)

    user_info = {
        "id": user_id,
        "username": username,
        "email": email,
        "created_at": user_row["created_at"]
    }

    return True, token, "Login successful.", user_info

def _record_failed_login(cursor, identifier: str, attempt_row, now_ts: float):
    current_attempts = (attempt_row["failed_attempts"] if attempt_row else 0) + 1
    locked_until = 0.0

    if current_attempts >= MAX_LOGIN_ATTEMPTS:
        locked_until = now_ts + LOCKOUT_SECONDS

    cursor.execute("""
        INSERT INTO login_attempts (identifier, failed_attempts, last_failed_at, locked_until)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(identifier) DO UPDATE SET
            failed_attempts = excluded.failed_attempts,
            last_failed_at = excluded.last_failed_at,
            locked_until = excluded.locked_until
    """, (identifier, current_attempts, now_ts, locked_until))

# --- Forgot Password: Route A (Security Question) ---

def get_security_question_for_user(
    username: str,
    db_path: Optional[str] = None
) -> Tuple[bool, Optional[str], str]:
    """
    Retrieves the security question stored during signup for the given username.
    """
    norm_user = username.strip()
    if not norm_user:
        return False, None, "Username is required."

    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT security_question
        FROM users
        WHERE username = ? COLLATE NOCASE
    """, (norm_user,))
    row = cursor.fetchone()
    conn.close()

    if not row:
        return False, None, "User not found."

    return True, row["security_question"], "Security question retrieved."

def reset_password_with_security_question(
    username: str,
    security_answer: str,
    new_password: str,
    confirm_password: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verifies user's security answer and updates password if complex and confirmed.
    """
    norm_user = username.strip()
    norm_answer = security_answer.strip()

    fields = {
        "Username": norm_user,
        "Security Answer": norm_answer,
        "New Password": new_password,
        "Confirm Password": confirm_password
    }
    is_valid, msg = validate_mandatory_fields(fields)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_complexity(new_password)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_confirmation(new_password, confirm_password)
    if not is_valid:
        return False, msg

    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, security_answer_hash
        FROM users
        WHERE username = ? COLLATE NOCASE
    """, (norm_user,))
    user_row = cursor.fetchone()

    if not user_row:
        conn.close()
        return False, "User not found."

    stored_answer_hash = user_row["security_answer_hash"]
    if not verify_security_answer(norm_answer, stored_answer_hash):
        conn.close()
        return False, "Incorrect security answer."

    new_hash = hash_secret(new_password)
    now_str = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(time.time()))

    cursor.execute("""
        UPDATE users
        SET password_hash = ?, updated_at = ?
        WHERE id = ?
    """, (new_hash, now_str, user_row["id"]))
    conn.commit()
    conn.close()

    return True, "Password has been successfully reset! You can now log in with your new password."

# --- Forgot Password: Route B (Email OTP) ---

def initiate_forgot_password_otp(
    email: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str, Optional[str]]:
    """
    Initiates password recovery via Email OTP for a registered email.
    """
    norm_email = email.strip().lower()
    is_valid, msg = validate_email_format(norm_email)
    if not is_valid:
        return False, msg, None

    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM users WHERE email = ? COLLATE NOCASE", (norm_email,))
    user_row = cursor.fetchone()
    conn.close()

    if not user_row:
        return False, "No registered account found with this email address.", None

    otp_ok, otp_code, otp_msg = create_and_store_otp(norm_email, "FORGOT_PASSWORD", db_path)
    if not otp_ok:
        return False, f"Failed to generate recovery OTP: {otp_msg}", None

    email_ok, email_msg = send_otp_email(norm_email, otp_code, "FORGOT_PASSWORD")

    status_note = f"Password recovery OTP sent to {norm_email}."
    if not email_ok:
        status_note += f" [Notice: {email_msg}]"

    return True, status_note, otp_code

def reset_password_with_otp(
    email: str,
    otp_code: str,
    new_password: str,
    confirm_password: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verifies the recovery OTP and updates the user's password.
    """
    norm_email = email.strip().lower()

    fields = {
        "Email": norm_email,
        "OTP Code": otp_code,
        "New Password": new_password,
        "Confirm Password": confirm_password
    }
    is_valid, msg = validate_mandatory_fields(fields)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_complexity(new_password)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_confirmation(new_password, confirm_password)
    if not is_valid:
        return False, msg

    # Verify 'FORGOT_PASSWORD' OTP
    is_valid, msg = verify_stored_otp(norm_email, "FORGOT_PASSWORD", otp_code, db_path)
    if not is_valid:
        return False, msg

    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM users WHERE email = ? COLLATE NOCASE", (norm_email,))
    user_row = cursor.fetchone()

    if not user_row:
        conn.close()
        return False, "User account not found."

    new_hash = hash_secret(new_password)
    now_str = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(time.time()))

    cursor.execute("""
        UPDATE users
        SET password_hash = ?, updated_at = ?
        WHERE id = ?
    """, (new_hash, now_str, user_row["id"]))
    conn.commit()
    conn.close()

    return True, "Password has been successfully reset! You can now log in with your new password."

# --- Admin Operations ---

def authenticate_admin(username: str, password: str) -> bool:
    """
    Authenticates administrative user against secure configuration.
    """
    if not username or not password:
        return False
    return (username.strip() == ADMIN_USERNAME) and (password.strip() == ADMIN_PASSWORD)

def get_all_registered_users(db_path: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Returns registered users catalog for Admin Dashboard.
    Strictly excludes password hashes and security answer hashes.
    """
    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, username, email, created_at, updated_at
        FROM users
        ORDER BY id ASC
    """)
    rows = cursor.fetchall()
    conn.close()

    users_list = []
    for r in rows:
        users_list.append({
            "User ID": r["id"],
            "Username": r["username"],
            "Email Address": r["email"],
            "Registration Date": r["created_at"],
            "Status": "Active"
        })
    return users_list


### Module: `src/app.py`
Streamlit UI application featuring bespoke light mode, Josefin Sans typography, and protected dashboards.

In [ ]:
%%writefile src/app.py
"""
Infosys Springboard Internship 7.0 - Batch 4
Milestone 1: User Authentication Module (Streamlit Application)

Bespoke editorial light-mode design featuring Josefin Sans typography,
tailored ink & cobalt accents, and clean human-designed layout hierarchy.

Features:
- Pure light-mode aesthetic (no dark mode, zero generic AI cliches)
- Distinctive typography using Josefin Sans (headings, actions, branding)
- User registration with mandatory Email OTP verification
- Multi-identifier authentication (Username or Email)
- HS256 JWT session tokens
- Dual-route password recovery (Security Question and Email OTP)
- Protected User and Administrator dashboards
- Strictly zero informal emojis or childish symbols
"""

import sys
import os
from pathlib import Path

# Add project root and src to path for direct running
CURRENT_DIR = Path(__file__).resolve().parent
if str(CURRENT_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR))
if str(CURRENT_DIR.parent) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR.parent))

import streamlit as st
import pandas as pd

try:
    from .config import SECURITY_QUESTIONS, EMAIL_ADDRESS, ADMIN_USERNAME
    from .database import init_db
    from .security import verify_jwt_token
    from .auth_service import (
        stage_signup,
        complete_signup,
        login_user,
        get_security_question_for_user,
        reset_password_with_security_question,
        initiate_forgot_password_otp,
        reset_password_with_otp,
        authenticate_admin,
        get_all_registered_users
    )
except ImportError:
    try:
        from src.config import SECURITY_QUESTIONS, EMAIL_ADDRESS, ADMIN_USERNAME
        from src.database import init_db
        from src.security import verify_jwt_token
        from src.auth_service import (
            stage_signup,
            complete_signup,
            login_user,
            get_security_question_for_user,
            reset_password_with_security_question,
            initiate_forgot_password_otp,
            reset_password_with_otp,
            authenticate_admin,
            get_all_registered_users
        )
    except ImportError:
        from config import SECURITY_QUESTIONS, EMAIL_ADDRESS, ADMIN_USERNAME
        from database import init_db
        from security import verify_jwt_token
        from auth_service import (
            stage_signup,
            complete_signup,
            login_user,
            get_security_question_for_user,
            reset_password_with_security_question,
            initiate_forgot_password_otp,
            reset_password_with_otp,
            authenticate_admin,
            get_all_registered_users
        )

# Initialize database
init_db()

# Streamlit Page Configuration
st.set_page_config(
    page_title="Infosys Springboard - Authentication Portal",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Bespoke Light-Mode Design System with Josefin Sans Typography
BESPOKE_LIGHT_CSS = """
<style>
    @import url('https://fonts.googleapis.com/css2?family=Josefin+Sans:ital,wght@0,300;0,400;0,500;0,600;0,700;1,400&family=Inter:wght@400;500;600&display=swap');

    /* Global Canvas & Light Theme */
    html, body, .stApp {
        background-color: #fafbfc !important;
        background-image: linear-gradient(180deg, #f1f4f8 0%, #fafbfc 280px) !important;
        font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
        color: #111827 !important;
    }

    header[data-testid="stHeader"] {
        background: transparent !important;
    }

    .block-container {
        padding-top: 2.2rem !important;
        padding-bottom: 3.5rem !important;
        max-width: 1020px !important;
    }

    /* Typography Hierarchy with Josefin Sans */
    h1, h2, h3, h4, h5, h6 {
        font-family: 'Josefin Sans', sans-serif !important;
        color: #0f172a !important;
        letter-spacing: -0.015em !important;
        font-weight: 600 !important;
    }

    p, span {
        font-family: 'Inter', sans-serif !important;
        color: #4b5563;
    }

    /* Handcrafted Card Container */
    .bespoke-card {
        background: #ffffff !important;
        border: 1px solid #e5e8eb !important;
        border-radius: 12px !important;
        padding: 34px 32px !important;
        box-shadow: 
            0 1px 3px rgba(0, 0, 0, 0.02),
            0 6px 18px -4px rgba(15, 23, 42, 0.04) !important;
        margin-bottom: 24px !important;
    }

    .bespoke-card-header {
        border-bottom: 1px solid #f1f3f5;
        padding-bottom: 18px;
        margin-bottom: 22px;
    }

    /* Editorial Pill Badges */
    .badge-editorial {
        display: inline-block;
        background-color: #f0f4f9;
        color: #0055b8;
        font-family: 'Josefin Sans', sans-serif;
        font-size: 11px;
        font-weight: 600;
        letter-spacing: 0.12em;
        text-transform: uppercase;
        padding: 4px 10px;
        border-radius: 4px;
        border: 1px solid #d4e0ee;
        margin-bottom: 12px;
    }

    .badge-verified-light {
        display: inline-block;
        background-color: #ecfdf5;
        color: #047857;
        font-family: 'Josefin Sans', sans-serif;
        font-size: 11px;
        font-weight: 600;
        letter-spacing: 0.12em;
        text-transform: uppercase;
        padding: 4px 10px;
        border-radius: 4px;
        border: 1px solid #a7f3d0;
        margin-bottom: 12px;
    }

    /* Form Inputs - Crisp & Defined */
    div[data-baseweb="input"] {
        background-color: #ffffff !important;
        border: 1px solid #d1d5db !important;
        border-radius: 8px !important;
        box-shadow: 0 1px 2px rgba(0, 0, 0, 0.03) !important;
        transition: border-color 0.15s ease, box-shadow 0.15s ease !important;
    }

    div[data-baseweb="input"]:hover {
        border-color: #9ca3af !important;
    }

    div[data-baseweb="input"]:focus-within {
        border-color: #0055b8 !important;
        box-shadow: 0 0 0 3px rgba(0, 85, 184, 0.12) !important;
    }

    input {
        color: #111827 !important;
        -webkit-text-fill-color: #111827 !important;
        font-family: 'Inter', sans-serif !important;
        font-size: 14.5px !important;
    }

    input::placeholder {
        color: #9ca3af !important;
        -webkit-text-fill-color: #9ca3af !important;
    }

    div[data-baseweb="select"] > div {
        background-color: #ffffff !important;
        border: 1px solid #d1d5db !important;
        border-radius: 8px !important;
        color: #111827 !important;
    }

    label[data-testid="stWidgetLabel"] p {
        font-family: 'Josefin Sans', sans-serif !important;
        font-size: 13.5px !important;
        font-weight: 600 !important;
        color: #374151 !important;
        margin-bottom: 4px !important;
        letter-spacing: 0.02em !important;
    }

    /* Bespoke Ink & Cobalt Buttons */
    div[data-testid="stButton"] > button {
        background-color: #0f172a !important;
        color: #ffffff !important;
        border: 1px solid #0f172a !important;
        border-radius: 8px !important;
        font-family: 'Josefin Sans', sans-serif !important;
        font-weight: 600 !important;
        font-size: 14px !important;
        letter-spacing: 0.06em !important;
        text-transform: uppercase !important;
        height: 44px !important;
        padding: 0 22px !important;
        box-shadow: 0 1px 2px rgba(15, 23, 42, 0.08), 0 2px 4px rgba(15, 23, 42, 0.04) !important;
        transition: all 0.18s ease-in-out !important;
        width: 100% !important;
        cursor: pointer !important;
    }

    div[data-testid="stButton"] > button:hover {
        background-color: #0055b8 !important;
        border-color: #0055b8 !important;
        transform: translateY(-1px) !important;
        box-shadow: 0 4px 12px rgba(0, 85, 184, 0.22) !important;
    }

    div[data-testid="stButton"] > button:active {
        transform: translateY(1px) !important;
        box-shadow: 0 1px 2px rgba(0, 85, 184, 0.15) !important;
    }

    /* Secondary Action Button */
    div.secondary-action > div[data-testid="stButton"] > button {
        background-color: #ffffff !important;
        color: #4b5563 !important;
        border: 1px solid #d1d5db !important;
        box-shadow: 0 1px 2px rgba(0, 0, 0, 0.03) !important;
    }

    div.secondary-action > div[data-testid="stButton"] > button:hover {
        background-color: #f3f4f6 !important;
        color: #111827 !important;
        border-color: #9ca3af !important;
        box-shadow: 0 2px 6px rgba(0, 0, 0, 0.06) !important;
    }

    /* Metric Panels */
    div[data-testid="stMetric"] {
        background: #ffffff !important;
        border: 1px solid #e5e8eb !important;
        border-radius: 10px !important;
        padding: 18px 20px !important;
        box-shadow: 0 1px 3px rgba(0, 0, 0, 0.02) !important;
    }

    div[data-testid="stMetricLabel"] p {
        font-family: 'Josefin Sans', sans-serif !important;
        color: #6b7280 !important;
        font-size: 11.5px !important;
        font-weight: 600 !important;
        text-transform: uppercase !important;
        letter-spacing: 0.08em !important;
    }

    div[data-testid="stMetricValue"] {
        font-family: 'Josefin Sans', sans-serif !important;
        color: #0f172a !important;
        font-weight: 700 !important;
        font-size: 26px !important;
    }

    /* Modern Minimalist Tabs */
    div[data-baseweb="tab-list"] {
        background: #f1f3f5 !important;
        border: 1px solid #e5e7eb !important;
        border-radius: 8px !important;
        padding: 4px !important;
        margin-bottom: 22px !important;
        gap: 4px !important;
    }

    button[data-baseweb="tab"] {
        font-family: 'Josefin Sans', sans-serif !important;
        color: #6b7280 !important;
        font-weight: 600 !important;
        font-size: 13.5px !important;
        border-radius: 6px !important;
        border: none !important;
        padding: 8px 18px !important;
        background: transparent !important;
        transition: all 0.15s ease !important;
    }

    button[data-baseweb="tab"]:hover {
        color: #111827 !important;
    }

    button[data-baseweb="tab"][aria-selected="true"] {
        background: #ffffff !important;
        color: #0055b8 !important;
        box-shadow: 0 1px 3px rgba(0, 0, 0, 0.08) !important;
        border: 1px solid rgba(0, 0, 0, 0.04) !important;
    }

    /* Sidebar Navigation */
    section[data-testid="stSidebar"] {
        background-color: #ffffff !important;
        border-right: 1px solid #e5e8eb !important;
    }

    div[data-testid="stRadio"] > div {
        gap: 4px !important;
    }

    div[data-testid="stRadio"] label {
        background: #fafbfc !important;
        border: 1px solid #e5e8eb !important;
        border-radius: 8px !important;
        padding: 9px 14px !important;
        margin-bottom: 4px !important;
        transition: all 0.15s ease !important;
    }

    div[data-testid="stRadio"] label:hover {
        background: #f0f4f9 !important;
        border-color: #d0deec !important;
    }

    div[data-testid="stRadio"] label p {
        font-family: 'Josefin Sans', sans-serif !important;
        color: #1f2937 !important;
        font-weight: 600 !important;
        font-size: 14px !important;
        letter-spacing: 0.02em !important;
    }

    /* Dataframe */
    div[data-testid="stDataFrame"] {
        border: 1px solid #e5e8eb !important;
        border-radius: 8px !important;
    }

    /* Alert Boxes */
    div[data-testid="stAlert"] {
        border-radius: 8px !important;
        font-family: 'Inter', sans-serif !important;
        font-size: 13.5px !important;
    }
</style>
"""
st.markdown(BESPOKE_LIGHT_CSS, unsafe_allow_html=True)

# Initialize Session State
if "jwt_token" not in st.session_state:
    st.session_state["jwt_token"] = None
if "user_info" not in st.session_state:
    st.session_state["user_info"] = None
if "admin_logged_in" not in st.session_state:
    st.session_state["admin_logged_in"] = False
if "signup_stage" not in st.session_state:
    st.session_state["signup_stage"] = "FORM"  # 'FORM' or 'OTP_VERIFY'
if "signup_email" not in st.session_state:
    st.session_state["signup_email"] = ""
if "forgot_route" not in st.session_state:
    st.session_state["forgot_route"] = "Security Question"

def logout_user():
    st.session_state["jwt_token"] = None
    st.session_state["user_info"] = None
    st.rerun()

def logout_admin():
    st.session_state["admin_logged_in"] = False
    st.rerun()

# --- SIDEBAR BRANDING & NAVIGATION ---
st.sidebar.markdown("""
<div style="padding: 12px 2px 20px 2px; border-bottom: 1px solid #f1f3f5; margin-bottom: 20px;">
    <div style="font-family: 'Josefin Sans', sans-serif; font-size: 11px; font-weight: 700; letter-spacing: 0.16em; color: #0055b8; text-transform: uppercase;">
        Infosys Springboard
    </div>
    <div style="font-family: 'Josefin Sans', sans-serif; font-size: 20px; font-weight: 700; color: #0f172a; margin-top: 4px; letter-spacing: -0.01em;">
        Identity Platform
    </div>
    <div style="font-family: 'Inter', sans-serif; font-size: 12px; color: #6b7280; margin-top: 2px;">
        Milestone 1 &bull; Batch 4
    </div>
</div>
""", unsafe_allow_html=True)

# Active session validation
is_user_authenticated = False
if st.session_state["jwt_token"]:
    is_valid, payload, _ = verify_jwt_token(st.session_state["jwt_token"])
    if is_valid and payload:
        is_user_authenticated = True
    else:
        st.session_state["jwt_token"] = None
        st.session_state["user_info"] = None

menu_options = []
if is_user_authenticated:
    menu_options = ["User Dashboard", "Administrator Portal"]
elif st.session_state["admin_logged_in"]:
    menu_options = ["Admin Management Console", "User Sign In"]
else:
    menu_options = ["Sign In", "Create Account", "Reset Password", "Administrator Portal"]

selected_page = st.sidebar.radio("Navigation", menu_options)

st.sidebar.markdown("---")
st.sidebar.markdown("""
<div style="font-size: 12px; color: #6b7280; line-height: 1.7; padding: 2px;">
    <strong style="font-family: 'Josefin Sans', sans-serif; color: #374151; letter-spacing: 0.04em;">SYSTEM ARCHITECTURE</strong><br>
    &bull; Single SQLite Data Store<br>
    &bull; Salted bcrypt Hashes<br>
    &bull; HS256 JWT Session Tokens<br>
    &bull; Mandatory Email OTP Signup<br>
    &bull; Isolated Recovery OTPs<br>
    &bull; Dual-Identifier Login<br>
    &bull; Rate-Limit & Lockout Protection
</div>
""", unsafe_allow_html=True)

# --- VIEW: USER DASHBOARD (PROTECTED) ---
if is_user_authenticated and selected_page in ["User Dashboard", "Sign In", "User Sign In"]:
    user = st.session_state["user_info"] or {}
    st.markdown("""
    <div class="bespoke-card">
        <span class="badge-verified-light">Authenticated Session &bull; JWT Verified</span>
        <h1 style="margin: 6px 0 8px 0; font-size: 28px;">Welcome, {}</h1>
        <p style="color: #6b7280; font-size: 14.5px; margin: 0;">
            Your active session is cryptographically secured using a JSON Web Token (HS256).
        </p>
    </div>
    """.format(user.get("username", "User")), unsafe_allow_html=True)

    col_profile, col_session = st.columns([2, 1])

    with col_profile:
        st.markdown("""
        <div class="bespoke-card">
            <div class="bespoke-card-header">
                <h3 style="margin: 0; font-size: 18px; color: #0f172a;">User Profile</h3>
            </div>
            <table style="width: 100%; border-collapse: collapse; font-size: 14px;">
                <tr style="border-bottom: 1px solid #f1f3f5;">
                    <td style="padding: 12px 0; color: #6b7280; width: 35%;">Username</td>
                    <td style="padding: 12px 0; font-weight: 600; color: #0f172a;">{}</td>
                </tr>
                <tr style="border-bottom: 1px solid #f1f3f5;">
                    <td style="padding: 12px 0; color: #6b7280;">Registered Email</td>
                    <td style="padding: 12px 0; font-weight: 600; color: #0f172a;">{}</td>
                </tr>
                <tr style="border-bottom: 1px solid #f1f3f5;">
                    <td style="padding: 12px 0; color: #6b7280;">System Identifier</td>
                    <td style="padding: 12px 0; color: #0055b8; font-family: monospace;">#{}</td>
                </tr>
                <tr>
                    <td style="padding: 12px 0; color: #6b7280;">Account Created</td>
                    <td style="padding: 12px 0; color: #374151;">{}</td>
                </tr>
            </table>
            <div style="margin-top: 22px; padding: 12px 14px; background: #f8fafc; border-radius: 6px; border: 1px solid #e2e8f0; font-size: 12.5px; color: #4b5563;">
                <strong style="color: #0055b8;">Security Status:</strong> Permanent account record verified. Passwords and security answers are stored strictly as salted bcrypt hashes.
            </div>
        </div>
        """.format(
            user.get("username", "N/A"),
            user.get("email", "N/A"),
            user.get("id", "N/A"),
            user.get("created_at", "N/A")
        ), unsafe_allow_html=True)

    with col_session:
        st.markdown("""
        <div class="bespoke-card">
            <div class="bespoke-card-header">
                <h3 style="margin: 0; font-size: 18px; color: #0f172a;">Session Control</h3>
            </div>
            <p style="font-size: 13px; color: #6b7280; line-height: 1.6; margin-bottom: 20px;">
                Token signature and claims are verified on each transaction. Terminate active credentials anytime.
            </p>
        </div>
        """, unsafe_allow_html=True)
        if st.button("Sign Out", key="user_logout_btn"):
            logout_user()

# --- VIEW: ADMIN DASHBOARD (PROTECTED) ---
elif st.session_state["admin_logged_in"] and selected_page in ["Admin Management Console", "Administrator Portal"]:
    st.markdown("""
    <div class="bespoke-card">
        <span class="badge-editorial">Administrative Access &bull; Privileged Console</span>
        <h1 style="margin: 6px 0 8px 0; font-size: 28px;">Administrator Directory</h1>
        <p style="color: #6b7280; font-size: 14.5px; margin: 0;">
            Catalog of all verified user accounts. Password hashes and security answer hashes are strictly masked.
        </p>
    </div>
    """, unsafe_allow_html=True)

    users_data = get_all_registered_users()
    stat1, stat2, stat3 = st.columns(3)
    with stat1:
        st.metric("Total Registered Users", len(users_data))
    with stat2:
        st.metric("Database File", "milestone1.db")
    with stat3:
        st.metric("Administrative User", ADMIN_USERNAME)

    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("<h4>Registered User Accounts</h4>", unsafe_allow_html=True)

    if users_data:
        df = pd.DataFrame(users_data)
        st.dataframe(df, use_container_width=True, hide_index=True)
    else:
        st.info("No user accounts registered yet. Complete user signup with email OTP verification to populate the directory.")

    st.markdown("<br>", unsafe_allow_html=True)
    col_out1, col_out2, col_out3 = st.columns([1, 1, 2])
    with col_out1:
        if st.button("Exit Admin Console", key="admin_exit_btn"):
            logout_admin()

# --- VIEW: ADMINISTRATOR LOGIN ---
elif selected_page == "Administrator Portal":
    st.markdown("""
    <div class="bespoke-card" style="max-width: 500px; margin: 0 auto;">
        <span class="badge-editorial">System Administration</span>
        <h2 style="margin: 6px 0 6px 0; font-size: 24px;">Administrator Sign In</h2>
        <p style="color: #6b7280; font-size: 13.5px; margin: 0;">
            Authenticate with dedicated administrative credentials.
        </p>
    </div>
    """, unsafe_allow_html=True)

    col_pad1, col_form, col_pad2 = st.columns([1, 2, 1])
    with col_form:
        with st.form("admin_login_form"):
            admin_user = st.text_input("Administrator Username", placeholder="e.g. admin")
            admin_pass = st.text_input("Administrator Password", type="password", placeholder="Enter admin password")
            admin_submit = st.form_submit_button("Authenticate")

            if admin_submit:
                if authenticate_admin(admin_user, admin_pass):
                    st.session_state["admin_logged_in"] = True
                    st.success("Administrative credentials verified.")
                    st.rerun()
                else:
                    st.error("Invalid administrator credentials.")

# --- VIEW: USER SIGN IN ---
elif selected_page in ["Sign In", "User Sign In"]:
    st.markdown("""
    <div class="bespoke-card" style="max-width: 500px; margin: 0 auto;">
        <span class="badge-editorial">Authentication</span>
        <h2 style="margin: 6px 0 6px 0; font-size: 24px;">Sign In to Account</h2>
        <p style="color: #6b7280; font-size: 13.5px; margin: 0;">
            Enter your credentials to access your protected dashboard.
        </p>
    </div>
    """, unsafe_allow_html=True)

    col_pad1, col_form, col_pad2 = st.columns([1, 2, 1])
    with col_form:
        with st.form("login_form"):
            identifier = st.text_input("Username or Email Address", placeholder="Enter registered username or email")
            password = st.text_input("Password", type="password", placeholder="Enter your password")
            login_submit = st.form_submit_button("Sign In")

            if login_submit:
                success, token, msg, user_info = login_user(identifier, password)
                if success and token:
                    st.session_state["jwt_token"] = token
                    st.session_state["user_info"] = user_info
                    st.success(msg)
                    st.rerun()
                else:
                    st.error(msg)

# --- VIEW: USER SIGN UP (MANDATORY EMAIL OTP GATED) ---
elif selected_page == "Create Account":
    st.markdown("""
    <div class="bespoke-card">
        <span class="badge-editorial">Registration Workflow</span>
        <h2 style="margin: 6px 0 6px 0; font-size: 24px;">Create New Account</h2>
        <p style="color: #6b7280; font-size: 13.5px; margin: 0;">
            Permanent accounts are committed to the database only after mandatory Email OTP verification.
        </p>
    </div>
    """, unsafe_allow_html=True)

    if st.session_state["signup_stage"] == "FORM":
        with st.form("signup_form"):
            col1, col2 = st.columns(2)
            with col1:
                username = st.text_input("Username*", placeholder="Minimum 3 characters, alphanumeric")
                email = st.text_input("Email Address*", placeholder="Standard format, e.g. ab@cd.ef")
                password = st.text_input("Password*", type="password", placeholder="8+ chars, upper, lower, digit, symbol")
                confirm_password = st.text_input("Confirm Password*", type="password", placeholder="Re-enter password")

            with col2:
                security_question = st.selectbox("Security Question*", SECURITY_QUESTIONS)
                security_answer = st.text_input("Security Answer*", placeholder="Answer for account recovery")
                st.caption("Your answer is hashed using salted bcrypt and used for secondary account recovery.")

            signup_submit = st.form_submit_button("Proceed to Email Verification")

            if signup_submit:
                ok, msg, dev_otp = stage_signup(
                    username=username,
                    email=email,
                    password=password,
                    confirm_password=confirm_password,
                    security_question=security_question,
                    security_answer=security_answer
                )
                if ok:
                    st.session_state["signup_stage"] = "OTP_VERIFY"
                    st.session_state["signup_email"] = email.strip().lower()
                    st.success(msg)
                    st.rerun()
                else:
                    st.error(msg)

    elif st.session_state["signup_stage"] == "OTP_VERIFY":
        col_pad1, col_otp, col_pad2 = st.columns([1, 2, 1])
        with col_otp:
            st.markdown(f"""
            <div class="bespoke-card">
                <span class="badge-editorial">Step 2 of 2 &bull; Verification</span>
                <h3 style="margin: 6px 0 10px 0; font-size: 20px;">Email OTP Verification</h3>
                <p style="font-size: 13.5px; color: #4b5563; margin-bottom: 0;">
                    A 6-digit verification code was dispatched to: <strong style="color: #0055b8;">{st.session_state["signup_email"]}</strong>.<br>
                    The permanent account record will be created only upon successful verification.
                </p>
            </div>
            """, unsafe_allow_html=True)

            with st.form("otp_verification_form"):
                otp_code = st.text_input("6-Digit Verification Code", max_chars=6, placeholder="Enter 6-digit code")
                verify_submit = st.form_submit_button("Verify and Activate Account")

                if verify_submit:
                    ok, msg = complete_signup(st.session_state["signup_email"], otp_code)
                    if ok:
                        st.session_state["signup_stage"] = "FORM"
                        st.session_state["signup_email"] = ""
                        st.success(msg)
                    else:
                        st.error(msg)

            st.markdown('<div class="secondary-action">', unsafe_allow_html=True)
            if st.button("Edit Registration Details", key="reset_signup_btn"):
                st.session_state["signup_stage"] = "FORM"
                st.session_state["signup_email"] = ""
                st.rerun()
            st.markdown('</div>', unsafe_allow_html=True)

# --- VIEW: FORGOT PASSWORD (DUAL ROUTE) ---
elif selected_page == "Reset Password":
    st.markdown("""
    <div class="bespoke-card">
        <span class="badge-editorial">Account Recovery</span>
        <h2 style="margin: 6px 0 6px 0; font-size: 24px;">Password Recovery</h2>
        <p style="color: #6b7280; font-size: 13.5px; margin: 0;">
            Recover access using either your preset Security Question or an Email One-Time Password.
        </p>
    </div>
    """, unsafe_allow_html=True)

    tab_sec_q, tab_otp = st.tabs(["Security Question", "Email Verification Code"])

    # ROUTE A: SECURITY QUESTION
    with tab_sec_q:
        st.markdown("<h5>Route A: Security Question Verification</h5>", unsafe_allow_html=True)
        with st.form("sec_q_fetch_form"):
            sq_user = st.text_input("Registered Username", key="sq_username_input", placeholder="Enter username")
            fetch_btn = st.form_submit_button("Lookup Security Question")

            if fetch_btn and sq_user:
                ok, question, msg = get_security_question_for_user(sq_user)
                if ok:
                    st.session_state["active_sec_q_user"] = sq_user
                    st.session_state["active_sec_question"] = question
                    st.success(f"Security Question: {question}")
                else:
                    st.error(msg)

        if "active_sec_question" in st.session_state and st.session_state.get("active_sec_q_user"):
            with st.form("sec_q_reset_form"):
                st.info(f"Question for {st.session_state['active_sec_q_user']}: {st.session_state['active_sec_question']}")
                sq_answer = st.text_input("Security Answer", type="password", placeholder="Enter answer")
                sq_new_pass = st.text_input("New Password", type="password", placeholder="8+ chars, upper, lower, digit, symbol")
                sq_confirm_pass = st.text_input("Confirm New Password", type="password", placeholder="Confirm new password")
                sq_reset_btn = st.form_submit_button("Update Password")

                if sq_reset_btn:
                    ok, msg = reset_password_with_security_question(
                        st.session_state["active_sec_q_user"],
                        sq_answer,
                        sq_new_pass,
                        sq_confirm_pass
                    )
                    if ok:
                        st.success(msg)
                        del st.session_state["active_sec_question"]
                        del st.session_state["active_sec_q_user"]
                    else:
                        st.error(msg)

    # ROUTE B: EMAIL OTP
    with tab_otp:
        st.markdown("<h5>Route B: Email Verification Code</h5>", unsafe_allow_html=True)
        with st.form("otp_recovery_dispatch_form"):
            rec_email = st.text_input("Registered Email Address", key="rec_email_input", placeholder="Enter email address")
            send_rec_otp_btn = st.form_submit_button("Send Recovery Code")

            if send_rec_otp_btn:
                ok, msg, dev_otp = initiate_forgot_password_otp(rec_email)
                if ok:
                    st.session_state["active_rec_email"] = rec_email.strip().lower()
                    st.success(msg)
                else:
                    st.error(msg)

        if "active_rec_email" in st.session_state:
            with st.form("otp_recovery_finalize_form"):
                st.info(f"Verification code dispatched to: {st.session_state['active_rec_email']}")
                rec_otp = st.text_input("6-Digit Verification Code", max_chars=6, placeholder="Enter 6-digit code")
                rec_new_pass = st.text_input("New Password", type="password", placeholder="8+ chars, upper, lower, digit, symbol")
                rec_confirm_pass = st.text_input("Confirm New Password", type="password", placeholder="Confirm new password")
                rec_reset_btn = st.form_submit_button("Verify Code and Update Password")

                if rec_reset_btn:
                    ok, msg = reset_password_with_otp(
                        st.session_state["active_rec_email"],
                        rec_otp,
                        rec_new_pass,
                        rec_confirm_pass
                    )
                    if ok:
                        st.success(msg)
                        del st.session_state["active_rec_email"]
                    else:
                        st.error(msg)


In [ ]:
%%writefile app.py
import sys
from pathlib import Path
src_dir = Path(__file__).resolve().parent / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
from src.app import *


### Automated Test Suite
Writes all unit and integration test files into `tests/`.

In [ ]:
%%writefile tests/test_validators.py
"""
Unit tests for input validation rules according to Infosys Milestone 1 requirements.
"""

import pytest
from src.validators import (
    validate_mandatory_fields,
    validate_email_format,
    validate_password_complexity,
    validate_password_confirmation,
    validate_username
)

def test_mandatory_fields_empty():
    fields = {"Username": "john", "Email": "", "Password": "SecretPassword1@"}
    valid, msg = validate_mandatory_fields(fields)
    assert not valid
    assert "Email" in msg

def test_mandatory_fields_whitespace():
    fields = {"Username": "  ", "Email": "john@example.com"}
    valid, msg = validate_mandatory_fields(fields)
    assert not valid
    assert "Username" in msg

def test_mandatory_fields_success():
    fields = {"Username": "john_doe", "Email": "john@example.com", "Password": "SecretPassword1@"}
    valid, msg = validate_mandatory_fields(fields)
    assert valid
    assert msg == ""

def test_email_format_infosys_rule():
    # Valid shape (at least 2 chars before @, between @ and dot, and after dot)
    valid_emails = [
        "ab@cd.ef",
        "john.doe@company.org",
        "intern2026@infosys.co.in",
        "user_123@domain.ac.uk"
    ]
    for email in valid_emails:
        valid, msg = validate_email_format(email)
        assert valid, f"Expected {email} to be valid, got: {msg}"

    # Invalid: less than 2 chars before @
    valid, msg = validate_email_format("a@cd.ef")
    assert not valid
    assert "at least 2 characters before" in msg

    # Invalid: less than 2 chars between @ and dot
    valid, msg = validate_email_format("ab@c.ef")
    assert not valid
    assert "at least 2 characters between" in msg

    # Invalid: less than 2 chars after final dot
    valid, msg = validate_email_format("ab@cd.e")
    assert not valid
    assert "at least 2 characters after" in msg

    # Invalid: missing @ or dot
    valid, _ = validate_email_format("plainaddress")
    assert not valid

    valid, _ = validate_email_format("user@nodot")
    assert not valid

def test_password_complexity():
    # Valid passwords: >=8 chars, uppercase, lowercase, digit, special symbol
    assert validate_password_complexity("Strong@123")[0]
    assert validate_password_complexity("Infosys#2026")[0]
    assert validate_password_complexity("P@ssw0rd!xyz")[0]

    # Too short (< 8 chars)
    valid, msg = validate_password_complexity("Ab1@")
    assert not valid
    assert "at least 8 characters" in msg

    # Missing uppercase
    valid, msg = validate_password_complexity("strong@123")
    assert not valid
    assert "uppercase" in msg

    # Missing lowercase
    valid, msg = validate_password_complexity("STRONG@123")
    assert not valid
    assert "lowercase" in msg

    # Missing digit
    valid, msg = validate_password_complexity("Strong@Password")
    assert not valid
    assert "numeric digit" in msg

    # Missing special character
    valid, msg = validate_password_complexity("StrongPass123")
    assert not valid
    assert "special symbol" in msg

def test_password_confirmation():
    valid, _ = validate_password_confirmation("Secret123!", "Secret123!")
    assert valid

    valid, msg = validate_password_confirmation("Secret123!", "Different123!")
    assert not valid
    assert "does not match" in msg

def test_username_format():
    assert validate_username("john_doe")[0]
    assert validate_username("user123")[0]
    assert validate_username("abc")[0]

    # Too short (<3 chars)
    valid, _ = validate_username("ab")
    assert not valid

    # Invalid characters (spaces, special symbols)
    valid, _ = validate_username("john doe")
    assert not valid

    valid, _ = validate_username("john@doe")
    assert not valid


In [ ]:
%%writefile tests/test_security.py
"""
Unit tests for security, bcrypt hashing, and JWT tokens.
"""

import time
import jwt
from src.security import (
    hash_secret,
    verify_secret,
    hash_security_answer,
    verify_security_answer,
    generate_jwt_token,
    verify_jwt_token
)
from src.config import JWT_SECRET, JWT_ALGORITHM

def test_bcrypt_hashing_and_verification():
    raw_password = "SecretPassword@123"
    hashed = hash_secret(raw_password)

    # Hash should be non-empty and start with $2b$
    assert hashed.startswith("$2b$")
    assert hashed != raw_password

    # Correct password verifies
    assert verify_secret(raw_password, hashed)

    # Wrong password fails
    assert not verify_secret("WrongPassword@123", hashed)

def test_security_answer_normalization():
    answer = "  Oxford University  "
    hashed = hash_security_answer(answer)

    # Check varying case and surrounding whitespace
    assert verify_security_answer("oxford university", hashed)
    assert verify_security_answer("OXFORD UNIVERSITY", hashed)
    assert verify_security_answer("   Oxford University   ", hashed)
    assert not verify_security_answer("Cambridge University", hashed)

def test_jwt_generation_and_verification():
    token = generate_jwt_token(user_id=42, username="test_user", email="test@example.com", expiry_hours=1)
    assert isinstance(token, str)

    is_valid, payload, msg = verify_jwt_token(token)
    assert is_valid
    assert payload is not None
    assert payload["sub"] == "42"
    assert payload["username"] == "test_user"
    assert payload["email"] == "test@example.com"
    assert msg == "Valid session."

def test_jwt_tampered_token():
    token = generate_jwt_token(user_id=1, username="alice", email="alice@example.com")
    tampered_token = token[:-5] + "abcde"

    is_valid, payload, msg = verify_jwt_token(tampered_token)
    assert not is_valid
    assert payload is None
    assert "Invalid session token" in msg

def test_jwt_expired_token():
    # Construct an expired token manually
    expired_payload = {
        "sub": "1",
        "username": "expired_user",
        "email": "exp@example.com",
        "iat": int(time.time()) - 7200,
        "exp": int(time.time()) - 3600
    }
    expired_token = jwt.encode(expired_payload, JWT_SECRET, algorithm=JWT_ALGORITHM)

    is_valid, payload, msg = verify_jwt_token(expired_token)
    assert not is_valid
    assert payload is None
    assert "Session has expired" in msg


In [ ]:
%%writefile tests/test_otp.py
"""
Unit tests for OTP generation, database hashing, attempt limits, expiry, and separation of purposes.
"""

import time
import os
import tempfile
import pytest
from src.database import init_db, get_connection
from src.otp_service import (
    generate_numeric_otp,
    create_and_store_otp,
    verify_stored_otp
)

@pytest.fixture
def temp_db():
    fd, path = tempfile.mkstemp(suffix=".db")
    os.close(fd)
    init_db(path)
    yield path
    if os.path.exists(path):
        os.remove(path)

def test_numeric_otp_format():
    otp = generate_numeric_otp(6)
    assert len(otp) == 6
    assert otp.isdigit()

def test_otp_creation_and_hashing(temp_db):
    email = "test@example.com"
    success, otp_code, msg = create_and_store_otp(email, "SIGNUP", temp_db)
    assert success
    assert len(otp_code) == 6

    # Verify that plaintext OTP is NOT in database
    conn = get_connection(temp_db)
    cursor = conn.cursor()
    cursor.execute("SELECT otp_hash FROM otp_codes WHERE identifier = ?", (email,))
    row = cursor.fetchone()
    conn.close()

    assert row is not None
    assert row["otp_hash"].startswith("$2b$")
    assert otp_code not in row["otp_hash"]

def test_otp_successful_verification(temp_db):
    email = "student@infosys.com"
    _, otp_code, _ = create_and_store_otp(email, "SIGNUP", temp_db)

    is_valid, msg = verify_stored_otp(email, "SIGNUP", otp_code, temp_db)
    assert is_valid
    assert "verified successfully" in msg.lower()

    # Once verified, it cannot be reused (one-time verification)
    is_valid_again, msg_again = verify_stored_otp(email, "SIGNUP", otp_code, temp_db)
    assert not is_valid_again
    assert "No active OTP request" in msg_again

def test_otp_separate_purposes(temp_db):
    # Ensure signup OTP and password-recovery OTP are separate
    email = "dual@example.com"
    _, signup_otp, _ = create_and_store_otp(email, "SIGNUP", temp_db)
    _, recovery_otp, _ = create_and_store_otp(email, "FORGOT_PASSWORD", temp_db)

    # Trying to verify signup OTP with FORGOT_PASSWORD purpose must fail
    is_valid, _ = verify_stored_otp(email, "FORGOT_PASSWORD", signup_otp, temp_db)
    assert not is_valid

    # Recovery OTP must verify with FORGOT_PASSWORD purpose
    is_valid, _ = verify_stored_otp(email, "FORGOT_PASSWORD", recovery_otp, temp_db)
    assert is_valid

def test_otp_attempt_limits(temp_db):
    email = "limit@example.com"
    _, real_otp, _ = create_and_store_otp(email, "SIGNUP", temp_db)

    # 1st wrong attempt
    valid, msg = verify_stored_otp(email, "SIGNUP", "000000", temp_db)
    assert not valid
    assert "2 attempt(s) remaining" in msg

    # 2nd wrong attempt
    valid, msg = verify_stored_otp(email, "SIGNUP", "111111", temp_db)
    assert not valid
    assert "1 attempt(s) remaining" in msg

    # 3rd wrong attempt: should invalidate
    valid, msg = verify_stored_otp(email, "SIGNUP", "222222", temp_db)
    assert not valid
    assert "Maximum attempts" in msg

    # Now even the correct OTP should be rejected because code was invalidated
    valid, msg = verify_stored_otp(email, "SIGNUP", real_otp, temp_db)
    assert not valid
    assert "No active OTP request" in msg

def test_otp_expired(temp_db):
    email = "expired@example.com"
    _, otp_code, _ = create_and_store_otp(email, "SIGNUP", temp_db)

    # Artificially expire the OTP in database
    conn = get_connection(temp_db)
    cursor = conn.cursor()
    cursor.execute("UPDATE otp_codes SET expires_at = ? WHERE identifier = ?", (time.time() - 100, email))
    conn.commit()
    conn.close()

    valid, msg = verify_stored_otp(email, "SIGNUP", otp_code, temp_db)
    assert not valid
    assert "expired" in msg.lower()


In [ ]:
%%writefile tests/test_auth.py
"""
Integration tests for complete authentication flows:
1. Signup staging & OTP-gated permanent account creation.
2. Login with username or email, brute-force lockout, and JWT token generation.
3. Forgot password Route A (Security question) & Route B (Email OTP).
4. Admin authentication and credential masking in user catalog.
"""

import os
import tempfile
import pytest
from src.database import init_db, get_connection
from src.config import SECURITY_QUESTIONS, ADMIN_USERNAME, ADMIN_PASSWORD
from src.security import verify_jwt_token
from src.auth_service import (
    stage_signup,
    complete_signup,
    login_user,
    get_security_question_for_user,
    reset_password_with_security_question,
    initiate_forgot_password_otp,
    reset_password_with_otp,
    authenticate_admin,
    get_all_registered_users
)

@pytest.fixture
def auth_db():
    fd, path = tempfile.mkstemp(suffix=".db")
    os.close(fd)
    init_db(path)
    yield path
    if os.path.exists(path):
        os.remove(path)

def test_signup_otp_gated_flow(auth_db):
    username = "alice_wonder"
    email = "alice@wonderland.org"
    password = "SecurePassword@123"
    question = SECURITY_QUESTIONS[0]
    answer = "Greenwood High"

    # 1. Stage signup
    ok, msg, dev_otp = stage_signup(
        username=username,
        email=email,
        password=password,
        confirm_password=password,
        security_question=question,
        security_answer=answer,
        db_path=auth_db
    )
    assert ok, f"Staging signup failed: {msg}"
    assert dev_otp is not None

    conn = get_connection(auth_db)
    cursor = conn.cursor()

    # 2. Verify account is NOT in permanent users table yet
    cursor.execute("SELECT * FROM users WHERE email = ?", (email,))
    assert cursor.fetchone() is None, "Permanent user account should NOT exist before OTP verification!"

    # 3. Verify record IS staged in pending_signups
    cursor.execute("SELECT * FROM pending_signups WHERE email = ?", (email,))
    staged = cursor.fetchone()
    assert staged is not None
    assert staged["username"] == username
    conn.close()

    # 4. Attempt login before OTP verification -> Must fail
    login_ok, _, _, _ = login_user(username, password, db_path=auth_db)
    assert not login_ok

    # 5. Complete signup with wrong OTP -> Fails
    comp_ok, comp_msg = complete_signup(email, "999999", db_path=auth_db)
    assert not comp_ok
    assert "Incorrect OTP" in comp_msg

    # 6. Complete signup with valid OTP -> Succeeds
    comp_ok, comp_msg = complete_signup(email, dev_otp, db_path=auth_db)
    assert comp_ok
    assert "successfully verified" in comp_msg.lower()

    # 7. Check permanent users table now contains user
    conn = get_connection(auth_db)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE email = ?", (email,))
    user = cursor.fetchone()
    assert user is not None
    assert user["username"] == username
    # Password hash must be bcrypt
    assert user["password_hash"].startswith("$2b$")
    assert user["security_answer_hash"].startswith("$2b$")

    # Pending signups record must be cleaned
    cursor.execute("SELECT * FROM pending_signups WHERE email = ?", (email,))
    assert cursor.fetchone() is None
    conn.close()

def test_login_with_username_and_email(auth_db):
    username = "bob_builder"
    email = "bob@builder.com"
    password = "BuilderPass@2026"
    question = SECURITY_QUESTIONS[1]
    answer = "Smith"

    # Stage and verify
    _, _, dev_otp = stage_signup(username, email, password, password, question, answer, db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    # 1. Login with username
    ok, token, msg, user_info = login_user(username, password, db_path=auth_db)
    assert ok
    assert token is not None
    assert user_info["username"] == username
    valid_jwt, payload, _ = verify_jwt_token(token)
    assert valid_jwt
    assert payload["username"] == username

    # 2. Login with email
    ok, token2, msg, user_info2 = login_user(email, password, db_path=auth_db)
    assert ok
    assert token2 is not None
    assert user_info2["email"] == email

    # 3. Wrong password -> generic error
    ok, token3, err_msg, _ = login_user(username, "WrongPassword@123", db_path=auth_db)
    assert not ok
    assert token3 is None
    assert err_msg == "Invalid username/email or password."

def test_login_brute_force_lockout(auth_db):
    username = "target_user"
    email = "target@example.com"
    password = "TargetPassword@123"

    _, _, dev_otp = stage_signup(username, email, password, password, SECURITY_QUESTIONS[0], "Answer", db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    # 3 consecutive failed logins
    for i in range(3):
        ok, _, msg, _ = login_user(username, f"WrongPassword@{i}", db_path=auth_db)
        assert not ok

    # 4th login attempt should report account locked
    ok, _, msg, _ = login_user(username, password, db_path=auth_db)
    assert not ok
    assert "locked" in msg.lower()

def test_forgot_password_route_a_security_question(auth_db):
    username = "charlie"
    email = "charlie@domain.com"
    password = "InitialPassword@123"
    question = SECURITY_QUESTIONS[2]
    answer = "Goldie"

    _, _, dev_otp = stage_signup(username, email, password, password, question, answer, db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    # Retrieve question
    ok, retrieved_q, _ = get_security_question_for_user(username, db_path=auth_db)
    assert ok
    assert retrieved_q == question

    # Wrong answer -> fails
    new_password = "UpdatedPassword@2026"
    ok, msg = reset_password_with_security_question(username, "WrongAnswer", new_password, new_password, db_path=auth_db)
    assert not ok
    assert "incorrect" in msg.lower()

    # Correct answer -> succeeds
    ok, msg = reset_password_with_security_question(username, "goldie", new_password, new_password, db_path=auth_db)
    assert ok

    # Verify login with new password
    login_ok, _, _, _ = login_user(username, new_password, db_path=auth_db)
    assert login_ok

def test_forgot_password_route_b_email_otp(auth_db):
    username = "diana"
    email = "diana@domain.com"
    password = "DianaPassword@123"

    _, _, dev_otp = stage_signup(username, email, password, password, SECURITY_QUESTIONS[0], "Ans", db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    # Request recovery OTP
    ok, msg, rec_otp = initiate_forgot_password_otp(email, db_path=auth_db)
    assert ok
    assert rec_otp is not None

    # Reset password with OTP
    new_pass = "DianaNewPassword@2026"
    ok, msg = reset_password_with_otp(email, rec_otp, new_pass, new_pass, db_path=auth_db)
    assert ok

    # Verify login with updated password
    login_ok, _, _, _ = login_user(email, new_pass, db_path=auth_db)
    assert login_ok

def test_admin_dashboard_security_and_masking(auth_db):
    # Verify admin credentials check
    assert authenticate_admin(ADMIN_USERNAME, ADMIN_PASSWORD)
    assert not authenticate_admin("admin", "WrongAdminPass")

    # Register sample user
    username = "eva_green"
    email = "eva@green.org"
    _, _, dev_otp = stage_signup(username, email, "EvaPass@2026", "EvaPass@2026", SECURITY_QUESTIONS[0], "Ans", db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    users_list = get_all_registered_users(db_path=auth_db)
    assert len(users_list) >= 1

    for u in users_list:
        assert "Username" in u
        assert "Email Address" in u
        # SECURITY REQUIREMENT: Never expose password hashes or security answer hashes
        assert "password" not in u
        assert "password_hash" not in u
        assert "security_answer" not in u
        assert "security_answer_hash" not in u


In [ ]:
# Step 4: Run Complete Automated Test Suite in Colab
!python -m pytest tests -v

### Launch Streamlit Portal with ngrok Tunnel
Run the cell below to launch the Streamlit application and establish a secure public ngrok tunnel.
Open the printed Public URL in your browser.

In [ ]:
# Step 5: Launch Streamlit & ngrok Tunnel
import subprocess
import time
from pyngrok import ngrok
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    import os
    NGROK_TOKEN = os.getenv('NGROK_AUTHTOKEN', '')

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print("[WARNING] NGROK_AUTHTOKEN is not configured. ngrok may fail without an authtoken.")

ngrok.kill()
!pkill -f streamlit

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

tunnel = ngrok.connect(8501)
print("=" * 65)
print(f"Infosys Portal Live URL: {tunnel.public_url}")
print("=" * 65)
print("Application is running. Press [Ctrl + C] or click Colab Interrupt to stop.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("[OK] Streamlit server and ngrok tunnel stopped gracefully.")